# (1) Giới thiệu

## Phân loại ảnh cảnh Intel — MobileNetV3-Small và ViT-B/16

So sánh hai mô hình pretrained trên sáu lớp buildings, forest, glacier, mountain, sea, street.
Hai câu hỏi: fine-tuning cải thiện validation bao nhiêu so với chỉ học classifier; checkpoint cuối phân loại test tốt đến đâu?
Notebook đi theo cùng 11 bước với mẫu Oxford Pets, dùng chung dữ liệu và protocol cho hai mô hình.

**Colab:** GPU T4, Internet, Run all. Mặc định Intel v2, seed 42, 3 epoch head, tối đa 15 epoch fine-tuning.
Đặt `cfg.smoke_test=True` hoặc biến môi trường `NOTEBOOK_SMOKE=1` để chạy 1 epoch mỗi giai đoạn trên subset đủ sáu lớp.
`RUN_TRAINING=False` chỉ xuất EDA. Lần full mới dùng `cfg.resume=False`; resume chỉ dùng checkpoint tương thích.
Checkpoint chọn hoàn toàn bằng validation macro-F1. Bảng, hình và kết luận lấy từ lượt chạy hiện tại.
Nguồn: [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification).


## 1.1. Thư viện

Giữ Torch/Torchvision có sẵn nếu đã cài. Cài thêm Grad-CAM cho phần giải thích cuối notebook.


In [ ]:
import importlib.util
import subprocess
import sys

if not all(importlib.util.find_spec(name) for name in ('torch', 'torchvision')):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch>=2.5,<3', 'torchvision>=0.20,<1'])
required = {
    'datasets': 'datasets>=3,<6', 'kagglehub': 'kagglehub>=0.3.12,<1',
    'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.2,<4',
    'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<1',
    'sklearn': 'scikit-learn>=1.5,<2',
    'cv2': 'opencv-python-headless>=4.10,<5', 'PIL': 'Pillow>=10,<13',
    'tqdm': 'tqdm>=4.66,<5', 'pytorch_grad_cam': 'grad-cam>=1.5,<2',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])


## 1.2. Import và cấu hình

Chỉ đổi cấu hình trước Run all; batch 16, accumulation 2, hai worker và seed 42.


In [ ]:
import gc
import hashlib
import io
import json
import math
import os
import platform
import random
import time
import textwrap
from contextlib import contextmanager, nullcontext
from dataclasses import asdict, dataclass
from importlib.metadata import version
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from datasets import Image as HFImage, load_dataset
import kagglehub
from IPython.display import display
from PIL import Image, ImageOps
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score,
                             roc_auc_score, top_k_accuracy_score)
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_info, threadpool_limits
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm  # progress bar dạng text để tránh backend widget trong notebook


In [ ]:
@dataclass
class Config:
    dataset_id: str = 'puneet6060/intel-image-classification'
    dataset_version: int | None = None  # None: bản mới nhất; đặt số version để cố định nguồn
    seed: int = 42
    smoke_test: bool = False
    smoke_train_per_class: int = 8
    smoke_val_per_class: int = 2
    smoke_test_per_class: int = 2
    val_fraction: float = 0.20
    batch_size: int = 16
    accumulation_steps: int = 2
    num_workers: int = 2  # giữ hai worker; iterator được đóng rõ ràng ở cuối mỗi lượt đọc
    head_epochs: int = 3
    finetune_epochs: int = 15
    head_lr: float = 1e-3
    backbone_lr: float = 1e-5
    finetune_head_lr: float = 1e-4
    weight_decay: float = 1e-4
    patience: int = 4
    resume: bool = False  # True: đọc last.pt trong cùng thư mục/config
    use_amp: bool = True
    cache_dir: str = './intel_cache'
    output_dir: str = './intel_transfer_results'
    cpu_threads: int = 2


cfg = Config(dataset_version=2, resume=False, smoke_test=os.environ.get('NOTEBOOK_SMOKE', '0') == '1')
LOCAL_IMAGE_DATASET_DIR = None  # Hoặc Path tới dữ liệu đã giải nén chứa seg_train/seg_test.
# cfg.smoke_test = True
# cfg.output_dir = '/kaggle/working/intel_transfer_results'
# cfg.resume = True  # cần giữ cả best_head.pt, best_finetune.pt và last.pt
RUN_TRAINING = True


## 1.3. Runtime và thư mục kết quả

Khóa seed, giới hạn thread CPU và lưu môi trường/cấu hình. Smoke và full dùng thư mục riêng.


In [ ]:
assert 0 < cfg.val_fraction < 1 and cfg.batch_size > 0 and cfg.accumulation_steps > 0
assert cfg.head_epochs > 0 and cfg.finetune_epochs > 0 and cfg.cpu_threads > 0
assert cfg.dataset_version is None or (isinstance(cfg.dataset_version, int) and cfg.dataset_version > 0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = cfg.use_amp and device.type == 'cuda'
torch.set_num_threads(cfg.cpu_threads)
cv2.setNumThreads(cfg.cpu_threads)
cpu_thread_limit = threadpool_limits(limits=cfg.cpu_threads)
ROOT = Path(cfg.output_dir) / ('smoke' if cfg.smoke_test else 'full')
CACHE = Path(cfg.cache_dir)
for directory in (ROOT, ROOT / 'figures', CACHE):
    directory.mkdir(parents=True, exist_ok=True)
# KaggleHub dùng cache này ngoài Kaggle; trong Kaggle có thể trả thư mục input được gắn sẵn.
os.environ['KAGGLEHUB_CACHE'] = str((CACHE / 'kagglehub').resolve())
torch.hub.set_dir(str(CACHE / 'torch'))
sns.set_theme(style='whitegrid')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def save_figure(name, caption=None):
    if cfg.smoke_test:
        plt.gcf().suptitle('SMOKE TEST — kết quả mô hình chỉ để kiểm chứng kỹ thuật', fontsize=11)
    figure = plt.gcf()
    if caption:
        wrapped_caption = textwrap.fill(caption, width=110)
        lines = wrapped_caption.count('\n') + 1
        bottom = min(0.30, 0.05 + lines * 0.14 / figure.get_figheight())
        figure.tight_layout(rect=(0, bottom, 1, 1))
        figure.text(0.5, 0.01, wrapped_caption, ha='center', va='bottom', fontsize=9)
    else:
        figure.tight_layout()
    plt.savefig(ROOT / 'figures' / f'{name}.png', dpi=160, bbox_inches='tight')
    plt.show()
    plt.close()
    if caption:
        print('Caption:', caption)

seed_everything(cfg.seed)
model_results = {}  # chỉ đăng ký mô hình thực sự đã được chạy
environment = {
    'python': sys.version, 'platform': platform.platform(), 'device': str(device),
    'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
    'cuda': torch.version.cuda, 'cpu_threads': torch.get_num_threads(),
    'packages': {name: version(name) for name in (
        'torch', 'torchvision', 'datasets', 'kagglehub', 'numpy', 'pandas',
        'matplotlib', 'seaborn', 'scikit-learn', 'Pillow')},
    'dataloader': {'num_workers': cfg.num_workers, 'persistent_workers': False,
                   'iterator_cleanup': 'owner_process_finally', 'progress_backend': 'tqdm.text'},
    'opencv': cv2.__version__,
    'opencv_threads': cv2.getNumThreads(), 'native_threadpools': threadpool_info(),
}
write_json(ROOT / 'environment.json', environment)
write_json(ROOT / 'config.json', asdict(cfg))
print('Thiết bị:', device, '| chế độ:', 'SMOKE TEST' if cfg.smoke_test else 'FULL')
if device.type == 'cpu':
    print('Không có CUDA: full ViT sẽ chậm. Có thể bật cfg.smoke_test để kiểm chứng pipeline.')


# (2) Tải và nạp dữ liệu

Tải Intel v2, chỉ dùng ảnh có nhãn trong seg_train/seg_test; ảnh seg_pred không tham gia đánh giá.


In [ ]:
dataset_handle = cfg.dataset_id
if cfg.dataset_version is not None:
    dataset_handle = f'{dataset_handle}/versions/{cfg.dataset_version}'
print('Nguồn Kaggle:', dataset_handle)
print('Tải/gắn toàn bộ dataset; chỉ nạp train/test sau khi dữ liệu đã sẵn sàng.')
if LOCAL_IMAGE_DATASET_DIR is None:
    dataset_dir = Path(kagglehub.dataset_download(dataset_handle)).resolve()
    download_method = 'kagglehub.dataset_download (entire dataset)'
else:
    dataset_dir = Path(LOCAL_IMAGE_DATASET_DIR).expanduser().resolve()
    download_method = 'local extracted dataset'
print('Thư mục dữ liệu cục bộ:', dataset_dir)

resolved_dataset_version = cfg.dataset_version
# Cache ngoài Kaggle thường kết thúc bằng /versions/N.
# Một số input trên Kaggle dùng /datasets/owner/slug/N.
version_in_path = None
if dataset_dir.name.isdecimal() and dataset_dir.parent.name in ('versions', cfg.dataset_id.rsplit('/', 1)[-1]):
    version_in_path = int(dataset_dir.name)
if version_in_path is not None:
    if cfg.dataset_version is not None and cfg.dataset_version != version_in_path:
        raise RuntimeError('Version trong đường dẫn dữ liệu khác version được yêu cầu.')
    resolved_dataset_version = version_in_path
print('Version:', resolved_dataset_version if resolved_dataset_version is not None else 'không xác định từ input; xem manifest dữ liệu')

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
def files_in(folder):
    roots = {p for p in dataset_dir.rglob(folder) if p.is_dir()}
    if dataset_dir.name == folder:
        roots.add(dataset_dir)
    # seg_train/seg_train: chỉ quét root ngoài cùng để không đếm ảnh hai lần.
    outer_roots = [root for root in roots if not any(parent in roots for parent in root.parents)]
    return sorted({
        p.relative_to(dataset_dir).as_posix()
        for root in outer_roots for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in image_extensions
    })

train_files, test_files, prediction_files = (files_in(folder) for folder in ('seg_train', 'seg_test', 'seg_pred'))
if not train_files or not test_files:
    raise RuntimeError('Không tìm thấy ảnh trong seg_train/seg_test dưới thư mục Kaggle đã tải.')
data_files = {
    split: [str(dataset_dir / p) for p in files]
    for split, files in (('train', train_files), ('test', test_files))
}
raw = load_dataset('imagefolder', data_files=data_files, cache_dir=str(CACHE / 'datasets'))
class_names = list(raw['train'].features['label'].names)
expected_names = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
assert class_names == expected_names, f'Nhãn ngoài dự kiến: {class_names}'
assert list(raw['test'].features['label'].names) == class_names
NUM_CLASSES = len(class_names)
LABELS = list(range(NUM_CLASSES))
raw = {split: ds.cast_column('image', HFImage(decode=False)) for split, ds in raw.items()}
source_info = {
    'provider': 'kaggle', 'dataset_id': cfg.dataset_id,
    'dataset_url': f'https://www.kaggle.com/datasets/{cfg.dataset_id}',
    'requested_handle': dataset_handle, 'requested_version': cfg.dataset_version,
    'resolved_version': resolved_dataset_version, 'class_names': class_names,
    'download_method': download_method,
    'local_dataset_dir': str(dataset_dir),
    'loaded_splits': {split: len(ds) for split, ds in raw.items()},
    'prediction_files_available': len(prediction_files),
    'fingerprints': {split: ds._fingerprint for split, ds in raw.items()},
    'test_policy': 'Giữ mọi ảnh test có nhãn hợp lệ và decode được; không deduplicate test.',
}
write_json(ROOT / 'dataset_source.json', source_info)
display(pd.DataFrame([{'split': k, 'images': v} for k, v in source_info['loaded_splits'].items()]))
print('Ảnh prediction trong dataset:', len(prediction_files), '(không dùng cho thí nghiệm có giám sát)')


# (3) Tiền xử lý và chia tập

Giải mã RGB, tính hash pixel để kiểm tra trùng và đo chất lượng ảnh gốc.


In [ ]:
def decode_image(record):
    payload = record['image']
    source = io.BytesIO(payload['bytes']) if payload.get('bytes') is not None else payload['path']
    with Image.open(source) as image:
        image.load()
        # Sửa orientation từ EXIF trước khi tính hash và mọi phép biến đổi khác.
        oriented = ImageOps.exif_transpose(image)
        original_mode = oriented.mode
        rgb = oriented.convert('RGB').copy()
    return rgb, original_mode


def image_from_row(row):
    return decode_image(raw[row['source_split']][int(row['row_index'])])[0]


def pixel_hash(image):
    digest = hashlib.sha256()
    digest.update(np.asarray(image.size, dtype=np.int64).tobytes())
    digest.update(image.tobytes())
    return digest.hexdigest()


## 3.1. Kiểm toán ảnh

Lưu log ảnh lỗi và kiểm tra nhãn. Giữ toàn bộ test hợp lệ; chỉ làm sạch dữ liệu phát triển.


In [ ]:
def audit_dataset(datasets):
    records = []
    for split, ds in datasets.items():
        for index in tqdm(range(len(ds)), desc=f'Kiểm tra {split}'):
            result = {'sample_id': f'{split}:{index}', 'source_split': split, 'row_index': index,
                      'label': -1, 'label_valid': False, 'decode_ok': False, 'error': ''}
            try:
                record = ds[index]
                label = record.get('label')
                valid = isinstance(label, (int, np.integer)) and 0 <= int(label) < NUM_CLASSES
                result.update(label=int(label) if valid else -1, label_valid=bool(valid))
                image, mode = decode_image(record)
                gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
                width, height = image.size
                result.update(decode_ok=True, width=width, height=height, aspect_ratio=width / height,
                              mode=mode, brightness=float(gray.mean()), contrast=float(gray.std()),
                              laplacian_var=float(cv2.Laplacian(gray, cv2.CV_64F).var()),
                              rgb_hash=pixel_hash(image))
            except Exception as exc:
                result['error'] = f'{type(exc).__name__}: {exc}'
            records.append(result)
    return pd.DataFrame(records)

audit = audit_dataset(raw)
audit.to_csv(ROOT / 'image_audit.csv', index=False)
valid = audit.loc[audit.decode_ok & audit.label_valid].copy()
valid['class_name'] = valid.label.map(dict(enumerate(class_names)))
excluded_invalid = audit.loc[~(audit.decode_ok & audit.label_valid)].copy()
excluded_invalid.to_csv(ROOT / 'invalid_images.csv', index=False)
display(audit.groupby('source_split').agg(total=('sample_id', 'size'),
                                        decodable=('decode_ok', 'sum'), valid_labels=('label_valid', 'sum')))
display(excluded_invalid.head(10))


## 3.2. Làm sạch và chia validation

Loại train trùng test, nhóm train mâu thuẫn nhãn và bản sao cùng nhãn; chia stratified 80/20, kiểm tra hash giữa mọi split.


In [ ]:
def clean_and_split(audit_frame):
    accepted = audit_frame[audit_frame.decode_ok & audit_frame.label_valid].copy()
    accepted['class_name'] = accepted.label.map(dict(enumerate(class_names)))
    test = accepted[accepted.source_split == 'test'].copy()
    train = accepted[accepted.source_split == 'train'].copy()
    test_hashes = set(test.rgb_hash)
    conflict_hashes = set(train.groupby('rgb_hash').label.nunique().loc[lambda x: x > 1].index)
    removals = []
    kept = []
    seen = set()
    for _, row in train.sort_values('row_index').iterrows():
        reason = None
        if row.rgb_hash in test_hashes:
            reason = 'train_duplicate_of_test'
        elif row.rgb_hash in conflict_hashes:
            reason = 'train_conflicting_labels'
        elif row.rgb_hash in seen:
            reason = 'train_exact_duplicate'
        if reason:
            record = row.to_dict()
            record['reason'] = reason
            removals.append(record)
        else:
            seen.add(row.rgb_hash)
            kept.append(row.to_dict())
    development = pd.DataFrame(kept)
    if development.empty or test.empty:
        raise ValueError('Không đủ dữ liệu hợp lệ sau kiểm tra chất lượng.')
    for name, frame in [('development', development), ('test', test)]:
        if set(frame.label) != set(LABELS):
            raise ValueError(f'{name} không còn đủ sáu lớp; cần kiểm tra log trước khi huấn luyện.')
    train_idx, val_idx = train_test_split(
        np.arange(len(development)), test_size=cfg.val_fraction,
        random_state=cfg.seed, stratify=development.label)
    splits = {'train': development.iloc[train_idx].copy(),
              'val': development.iloc[val_idx].copy(), 'test': test}
    return splits, pd.DataFrame(removals, columns=list(train.columns) + ['reason'])

splits_full, cleaning_log = clean_and_split(audit)
cleaning_log.to_csv(ROOT / 'cleaning_log.csv', index=False)
display(cleaning_log.groupby(['source_split', 'reason']).size().rename('count').reset_index())

def stratified_subset(frame, per_class):
    parts = []
    for label in LABELS:
        group = frame[frame.label == label]
        if len(group) < per_class:
            raise ValueError(f'Lớp {class_names[label]} chỉ có {len(group)} ảnh, cần {per_class} cho smoke test.')
        parts.append(group.sample(per_class, random_state=cfg.seed + label))
    return pd.concat(parts).sort_values('sample_id').reset_index(drop=True)

splits = {name: frame.reset_index(drop=True) for name, frame in splits_full.items()}
if cfg.smoke_test:
    splits = {name: stratified_subset(splits[name], size) for name, size in [
        ('train', cfg.smoke_train_per_class), ('val', cfg.smoke_val_per_class),
        ('test', cfg.smoke_test_per_class)]}

def verify_splits(frames):
    for name, frame in frames.items():
        assert len(frame) > 0 and set(frame.label) == set(LABELS), name
        assert frame.sample_id.is_unique
    for left, right in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
        assert not set(frames[left].sample_id) & set(frames[right].sample_id)
        assert not set(frames[left].rgb_hash) & set(frames[right].rgb_hash)

verify_splits(splits_full)
verify_splits(splits)
manifest_full = pd.concat([frame.assign(split=name) for name, frame in splits_full.items()], ignore_index=True)
manifest = pd.concat([frame.assign(split=name) for name, frame in splits.items()], ignore_index=True)
manifest_full.to_csv(ROOT / 'split_manifest_full.csv', index=False)
manifest.to_csv(ROOT / 'split_manifest.csv', index=False)
split_counts = pd.crosstab(manifest.class_name, manifest.split).reindex(class_names)
display(split_counts)
split_counts.to_csv(ROOT / 'class_counts_used.csv')


## 3.3. Transform và Dataset

Train có augmentation; validation/test dùng preprocessing xác định của đúng pretrained weights.


In [ ]:
WEIGHTS = {
    'mobilenet_v3_small': models.MobileNet_V3_Small_Weights.IMAGENET1K_V1,
    'vit_b_16': models.ViT_B_16_Weights.IMAGENET1K_V1,
}

def training_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0), interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])


class SceneDataset(Dataset):
    def __init__(self, frame, transform):
        self.rows = frame.to_dict('records')
        self.transform = transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        return self.transform(image_from_row(row)), int(row['label']), row['sample_id']


# (4) EDA — dữ liệu gốc và đầu vào mô hình

EDA vẫn dùng toàn bộ ảnh trước subset smoke. Các chỉ số chất lượng chỉ gợi ý trường hợp cần xem bằng mắt; resize không tạo thêm chi tiết.


In [ ]:
eda_dir = ROOT / 'eda'
eda_dir.mkdir(exist_ok=True)

def save_eda_figure(figure, filename):
    figure.tight_layout()
    figure.savefig(eda_dir / filename, dpi=140, bbox_inches='tight')
    plt.show()
    plt.close(figure)

counts = pd.crosstab(valid.class_name, valid.source_split).reindex(class_names, fill_value=0)
counts.to_csv(ROOT / 'class_counts_raw.csv')

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
sample_rows = []
for label, ax in enumerate(axes.flat):
    rows = valid[(valid.source_split == 'train') & (valid.label == label)]
    if len(rows):
        row = rows.sample(1, random_state=cfg.seed).iloc[0]
        sample_rows.append(row.sample_id)
        ax.imshow(image_from_row(row))
        ax.set_title(f'{class_names[label]} | {row.width:.0f}×{row.height:.0f} px')
    ax.axis('off')
pd.Series(sample_rows, name='sample_id').to_csv(eda_dir / 'representative_samples.csv', index=False)
save_eda_figure(fig, 'eda-samples.png')


In [ ]:
def geometry_summary(frame):
    data = frame.assign(short_edge=frame[['width', 'height']].min(axis=1))
    summary = data.groupby('source_split')['short_edge'].agg(
        samples='size', minimum='min', median='median', below_128=lambda values: values.lt(128).sum(),
    )
    summary['below_128_pct'] = 100 * summary['below_128'] / summary['samples']
    return summary

geometry = geometry_summary(valid)
geometry.to_csv(eda_dir / 'short_edge_summary.csv')
display(geometry.round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, field, xlabel in zip(axes.flat[:3], ('width', 'height', 'aspect_ratio'),
                            ('Chiều rộng (px)', 'Chiều cao (px)', 'Tỷ lệ rộng/cao')):
    sns.histplot(data=valid, x=field, hue='source_split', bins=40, element='step', ax=ax)
    ax.set(xlabel=xlabel, ylabel='Số ảnh')
bars = axes[1, 1].bar(geometry.index, geometry.below_128_pct, color='#3d749e')
axes[1, 1].bar_label(bars, labels=[
    f'{row.below_128:.0f}/{row.samples:.0f}' for row in geometry.itertuples()
], padding=3)
axes[1, 1].set(ylabel='Tỷ lệ (%)', title='Cạnh ngắn dưới 128 px',
               ylim=(0, max(0.1, geometry.below_128_pct.max()) * 1.2))
save_eda_figure(fig, 'eda-geometry.png')

measurements = ('brightness', 'contrast', 'laplacian_var')
quality_summary = valid.groupby('source_split')[list(measurements)].describe()
quality_summary.to_csv(eda_dir / 'quality_summary.csv')
display(quality_summary)
color_counts = pd.crosstab(valid['mode'], valid.source_split)
color_counts.to_csv(eda_dir / 'color_modes.csv')
display(color_counts)

quality = valid.assign(log_laplacian=np.log1p(valid.laplacian_var))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, field, xlabel in zip(axes, ('brightness', 'contrast', 'log_laplacian'),
                            ('Brightness (mean pixel)', 'Contrast (std pixel)', 'log(1 + Laplacian variance)')):
    sns.histplot(data=quality, x=field, hue='source_split', bins=40, element='step', ax=ax)
    ax.set(xlabel=xlabel, ylabel='Số ảnh')
save_eda_figure(fig, 'eda-quality.png')

train_audit = valid[valid.source_split == 'train']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
extremes = []
for ax, (column, smallest) in zip(axes.flat, [
    ('brightness', True), ('brightness', False), ('contrast', True),
    ('laplacian_var', True), ('laplacian_var', False), ('aspect_ratio', False)]):
    rows = train_audit.sort_values([column, 'sample_id'], ascending=[smallest, True])
    if len(rows):
        row = rows.iloc[0]
        extremes.append({'sample_id': row.sample_id, 'measure': column, 'value': row[column],
                         'extreme': 'min' if smallest else 'max'})
        ax.imshow(image_from_row(row))
        ax.set_title(f'{row.class_name}\n{column}={row[column]:.2f}')
    ax.axis('off')
pd.DataFrame(extremes).to_csv(eda_dir / 'extreme_samples.csv', index=False)
save_eda_figure(fig, 'eda-extremes.png')


In [ ]:
duplicate_rows = valid[valid.duplicated('rgb_hash', keep=False)].sort_values(['rgb_hash', 'sample_id'])
duplicate_rows.to_csv(ROOT / 'duplicates.csv', index=False)
duplicate_summary = (duplicate_rows.groupby('rgb_hash').agg(
    count=('sample_id', 'size'), labels=('label', 'nunique'),
    source_splits=('source_split', lambda values: ','.join(sorted(set(values))))).reset_index())
display(duplicate_summary.head(10))


In [ ]:
split_plot_counts = pd.DataFrame({
    'Train gốc hợp lệ': counts['train'],
    'Test gốc hợp lệ': counts['test'],
    **{name: frame.class_name.value_counts().reindex(class_names, fill_value=0)
       for name, frame in splits_full.items()},
})
split_plot_counts.to_csv(eda_dir / 'class_counts.csv')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
split_plot_counts.iloc[:, :2].plot.bar(ax=axes[0], rot=15)
split_plot_counts.iloc[:, 2:].plot.bar(ax=axes[1], rot=15)
for ax, title in zip(axes, ('Ảnh hợp lệ trước lọc trùng', 'Ảnh sau làm sạch và chia tập')):
    ax.set(title=title, xlabel='', ylabel='Số ảnh')
save_eda_figure(fig, 'eda-class-counts.png')

reasons = {
    'train_duplicate_of_test': 'Trùng pixel với test',
    'train_conflicting_labels': 'Mâu thuẫn nhãn trong train',
    'train_exact_duplicate': 'Bản sao cùng nhãn trong train',
}
removed_counts = cleaning_log.reason.value_counts().reindex(list(reasons), fill_value=0)
removed_counts.rename('rows').to_csv(eda_dir / 'duplicate_removal_counts.csv')
fig, axes = plt.subplots(1, 3, figsize=(13, 4), gridspec_kw={'width_ratios': [2.0, 1, 1]})
stat_ax = axes[0]
bars = stat_ax.barh(list(reasons.values()), removed_counts.to_numpy(), color='#3d749e')
stat_ax.bar_label(bars, padding=3)
stat_ax.set(xlim=(0, max(1, removed_counts.max()) * 1.2), xlabel='Ảnh train bị loại')
example_ids = []
if not duplicate_summary.empty:
    example_hash = duplicate_summary.sort_values(['labels', 'rgb_hash'], ascending=[False, True]).iloc[0].rgb_hash
    group = duplicate_rows.loc[duplicate_rows.rgb_hash == example_hash]
    examples = (group.drop_duplicates('label') if group.label.nunique() > 1 else group).head(2)
    for ax, row in zip(axes[1:], examples.to_dict('records')):
        ax.imshow(image_from_row(row))
        ax.set_title(f"{row['sample_id']}\n{row['class_name']}")
        example_ids.append(row['sample_id'])
else:
    axes[1].text(0.5, 0.5, 'Không có nhóm trùng pixel', ha='center', va='center')
for ax in axes[1:]:
    ax.axis('off')
pd.Series(example_ids, name='sample_id').to_csv(eda_dir / 'duplicate_examples.csv', index=False)
save_eda_figure(fig, 'eda-duplicates.png')


## 4.1. Bằng chứng cho preprocessing và định danh thí nghiệm

Lưu đề xuất từ EDA và chữ ký dữ liệu/cấu hình để checkpoint không bị dùng nhầm.


In [ ]:
recommendations = []
def recommend(problem, evidence, action):
    recommendations.append({'Vấn đề/quan sát': problem, 'Bằng chứng': evidence, 'Đề xuất': action})

if len(excluded_invalid):
    recommend('Ảnh lỗi hoặc nhãn không hợp lệ', f'{len(excluded_invalid)} ảnh; xem invalid_images.csv',
              'Loại khỏi split tương ứng và báo số lượng; không tự sửa nhãn test.')
if len(cleaning_log):
    recommend('Ảnh trùng ảnh hưởng tập phát triển', f'{len(cleaning_log)} ảnh train bị loại; xem cleaning_log.csv',
              'Làm sạch trước chia validation; kiểm tra thủ công nhóm nhãn mâu thuẫn.')
test_duplicates = int(splits_full['test'].duplicated('rgb_hash').sum())
if test_duplicates:
    recommend('Bản sao trong test', f'{test_duplicates} bản sao ngoài ảnh đầu tiên mỗi hash',
              'Giữ test gốc, ghi rõ số mẫu độc lập có thể thấp hơn số ảnh.')
dev = pd.concat([splits_full['train'], splits_full['val']], ignore_index=True)
class_count = dev.groupby('label').size().reindex(LABELS, fill_value=0)
imbalance_ratio = float(class_count.max() / class_count.min())
recommend('Độ cân bằng lớp', f'max/min={imbalance_ratio:.2f}',
          'Thử class-weight hoặc sampler chỉ trên train, chọn bằng validation.' if imbalance_ratio > 1.5
          else 'Dùng cross-entropy thường và báo macro-F1; chưa cần tự cân bằng lớp.')
non_rgb = int((dev['mode'] != 'RGB').sum())
recommend('Kích thước/chế độ màu', f'{non_rgb} ảnh không phải RGB; median={dev.width.median():.0f}×{dev.height.median():.0f}',
          'Chuyển RGB, resize/crop theo pretrained weights; upsample không tạo thêm chi tiết.')
dark = int((dev.brightness < 40).sum())
low_contrast = int((dev.contrast < 15).sum())
if dark or low_contrast:
    recommend('Ảnh tối/ít tương phản cần kiểm tra', f'{dark} ảnh brightness<40; {low_contrast} ảnh std<15',
              'Xem ảnh mẫu; có thể thử CLAHE trên train/validation như ablation nếu cải thiện; giữ pipeline gốc làm mốc.')
flat = int((dev.laplacian_var < 10).sum())
if flat:
    recommend('Ảnh ít cạnh/độ sắc nét thấp', f'{flat} ảnh Laplacian variance<10 (chỉ ngưỡng gợi ý)',
              'Kiểm tra bằng mắt. Chỉ thử khử nhiễu khi thấy nhiễu thực tế; không tự xóa ảnh biển hoặc ảnh ít texture.')
recommend('Nhãn mơ hồ và gần trùng', 'Hash pixel không phát hiện near-duplicate; độ tin cậy không xác nhận lỗi nhãn',
          'Kiểm tra thủ công các nhóm dễ nhầm và dự đoán sai; cân nhắc perceptual hash/group split trong nghiên cứu tiếp theo.')
recommendations = pd.DataFrame(recommendations)
display(recommendations)
recommendations.to_csv(ROOT / 'preprocessing_recommendations.csv', index=False)

manifest_digest = hashlib.sha256(manifest[['sample_id', 'label', 'rgb_hash', 'split']].to_csv(index=False).encode()).hexdigest()
experiment = {'config': {k: v for k, v in asdict(cfg).items() if k not in ('resume', 'output_dir', 'cache_dir')},
              'source': {'provider': 'kaggle', 'dataset_id': cfg.dataset_id, 'version': resolved_dataset_version},
              'manifest_digest': manifest_digest,
              'weights': {'mobilenet_v3_small': 'IMAGENET1K_V1', 'vit_b_16': 'IMAGENET1K_V1'},
              'pipeline_version': 3, 'finetune_start': 'best_head', 'tie_policy': 'head'}
RUN_SIGNATURE = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()
write_json(ROOT / 'experiment.json', {**experiment, 'run_signature': RUN_SIGNATURE})


## 4.2. Xem ảnh sau transform

So sánh ảnh gốc với đầu vào crop/resize thực tế để hiểu phần nội dung mô hình được nhìn thấy.


In [ ]:
def plot_transform_preview():
    chosen = [splits_full['train'].loc[lambda f: f.class_name == label]
              .sample(1, random_state=cfg.seed).iloc[0] for label in ('glacier', 'mountain')]
    eval_recipe = WEIGHTS['mobilenet_v3_small'].transforms()
    train_recipe = training_transform()
    mean = torch.tensor(eval_recipe.mean).view(3, 1, 1)
    std = torch.tensor(eval_recipe.std).view(3, 1, 1)
    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(cfg.seed)
        for row, panel in zip(chosen, axes):
            image = image_from_row(row)
            augmented = train_recipe(image)
            panel[0].imshow(image)
            for ax, tensor in zip(panel[1:], (eval_recipe(image), augmented)):
                assert tuple(tensor.shape) == (3, 224, 224) and torch.isfinite(tensor).all()
                ax.imshow((tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
            for ax, title in zip(panel, ('Ảnh gốc', 'Validation', 'Train augmentation')):
                ax.set_title(f'{row.class_name} · {title}')
                ax.axis('off')
    pd.DataFrame(chosen)[['sample_id', 'class_name']].to_csv(eda_dir / 'transform_examples.csv', index=False)
    save_eda_figure(fig, 'eda-transforms.png')

plot_transform_preview()
write_json(eda_dir / 'eda_source.json', {
    'dataset_id': cfg.dataset_id,
    'dataset_version': resolved_dataset_version,
    'seed': cfg.seed,
    'manifest_digest': manifest_digest,
    'figures': sorted(path.name for path in eda_dir.glob('*.png')),
})
print('Đã xuất EDA tại', eda_dir.resolve())


# (5) Xây dựng mô hình

Dùng pretrained ImageNet, thay head sáu lớp. Hai mô hình được khởi tạo riêng khi đến lượt để không giữ đồng thời trên GPU.


In [ ]:
def build_model(model_name, pretrained=True):
    weights = WEIGHTS[model_name] if pretrained else None
    if model_name == 'mobilenet_v3_small':
        model = models.mobilenet_v3_small(weights=weights)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)
    elif model_name == 'vit_b_16':
        model = models.vit_b_16(weights=weights)
        model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
    else:
        raise ValueError(model_name)
    return model


def classifier(model, model_name):
    return model.classifier if model_name == 'mobilenet_v3_small' else model.heads


def is_head_key(key, model_name):
    return key.startswith('classifier.' if model_name == 'mobilenet_v3_small' else 'heads.')


def configure_stage(model, model_name, stage):
    for parameter in model.parameters():
        parameter.requires_grad = stage == 'finetune'
    for parameter in classifier(model, model_name).parameters():
        parameter.requires_grad = True


def set_training_mode(model, model_name, stage):
    if stage == 'head':
        model.eval()  # bao gồm BatchNorm của backbone: không cập nhật running stats
        classifier(model, model_name).train()
    else:
        model.train()


# (6) Huấn luyện mô hình

Giữ backbone ở eval khi chỉ học head; fine-tuning bắt đầu từ best head. Chọn best giữa hai giai đoạn bằng validation macro-F1, hòa điểm ưu tiên head. Checkpoint lưu RNG/optimizer/scheduler để resume đúng protocol.


In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


@contextmanager
def managed_loader_iterator(loader):
    # Dọn iterator cũ không còn được tham chiếu trong tiến trình chính trước
    # khi DataLoader tạo nhóm worker mới, để chúng không bị kế thừa qua fork.
    gc.collect()
    owner_pid = os.getpid()
    iterator = iter(loader)
    try:
        yield iterator
    finally:
        # PyTorch chưa cung cấp public close() cho DataLoader iterator.
        # Chỉ tiến trình tạo iterator được yêu cầu dừng/join các worker.
        if os.getpid() == owner_pid:
            shutdown = getattr(iterator, '_shutdown_workers', None)
            if callable(shutdown):
                shutdown()
        del iterator


def make_loaders(model_name):
    generator = torch.Generator().manual_seed(cfg.seed)
    loaders = {}
    for split in ('train', 'val'):
        transform = training_transform() if split == 'train' else WEIGHTS[model_name].transforms()
        loaders[split] = DataLoader(
            SceneDataset(splits[split], transform), batch_size=cfg.batch_size,
            shuffle=split == 'train', drop_last=False, num_workers=cfg.num_workers,
            persistent_workers=False,
            pin_memory=device.type == 'cuda', worker_init_fn=seed_worker,
            generator=generator if split == 'train' else torch.Generator().manual_seed(cfg.seed),
        )
    return loaders, generator


In [ ]:
def metric_values(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)),
        'macro_recall': float(recall_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, labels=LABELS, average='weighted', zero_division=0)),
    }


@torch.inference_mode()
def evaluate_neural(model, loader, target_device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    truth, predictions, probabilities, sample_ids = [], [], [], []
    with managed_loader_iterator(loader) as batches:
        for x, y, ids in batches:
            x, y = x.to(target_device), y.to(target_device)
            logits = model(x)  # eval FP32 để kiểm chứng checkpoint và báo metrics
            total_loss += F.cross_entropy(logits, y, reduction='sum').item()
            total_samples += len(y)
            truth.extend(y.cpu().tolist())
            predictions.extend(logits.argmax(1).cpu().tolist())
            probabilities.extend(logits.softmax(1).cpu().tolist())
            sample_ids.extend(ids)
    return {'loss': total_loss / total_samples, **metric_values(truth, predictions)}, \
           pd.DataFrame({'sample_id': sample_ids, 'label': truth, 'prediction': predictions}), \
           np.asarray(probabilities)


In [ ]:
def rng_state():
    return {'python': random.getstate(), 'numpy': np.random.get_state(), 'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if torch.cuda.is_available() and state['cuda'] is not None:
        torch.cuda.set_rng_state_all([value.cpu() for value in state['cuda']])


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_suffix('.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def backbone_digest(model, model_name):
    digest = hashlib.sha256()
    for key, tensor in model.state_dict().items():
        if not is_head_key(key, model_name):
            digest.update(key.encode())
            digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def load_checkpoint(path, model_name):
    state = torch.load(path, map_location='cpu', weights_only=False)
    if state['run_signature'] != RUN_SIGNATURE or state['model_name'] != model_name:
        raise ValueError('Checkpoint khác model/dữ liệu/config. Dùng output_dir mới hoặc checkpoint tương thích.')
    return state


In [ ]:
def train_epoch(model, model_name, stage, loader, optimizer, scaler):
    set_training_mode(model, model_name, stage)
    optimizer.zero_grad(set_to_none=True)
    truth, predictions = [], []
    total_loss, sample_count = 0.0, 0
    digest_before = backbone_digest(model, model_name) if cfg.smoke_test else None
    with managed_loader_iterator(loader) as batches:
        with tqdm(total=len(loader), desc=f'{model_name}/{stage}', leave=False) as progress:
            for batch_index, (x, y, _) in enumerate(batches):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                window_start = (batch_index // cfg.accumulation_steps) * cfg.accumulation_steps * loader.batch_size
                window_samples = min(cfg.accumulation_steps * loader.batch_size, len(loader.dataset) - window_start)
                context = torch.autocast(device_type='cuda', dtype=torch.float16) if AMP_ENABLED else nullcontext()
                with context:
                    logits = model(x)
                    loss = F.cross_entropy(logits, y)
                    accumulated_loss = loss * (len(y) / window_samples)
                scaler.scale(accumulated_loss).backward()
                if (batch_index + 1) % cfg.accumulation_steps == 0 or batch_index + 1 == len(loader):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                total_loss += loss.item() * len(y)
                sample_count += len(y)
                truth.extend(y.detach().cpu().tolist())
                predictions.extend(logits.detach().argmax(1).cpu().tolist())
                progress.update(1)
    if cfg.smoke_test:
        changed = digest_before != backbone_digest(model, model_name)
        assert changed == (stage == 'finetune'), f'Backbone freeze/unfreeze sai ở {model_name}/{stage}'
    return {'loss': total_loss / sample_count, **metric_values(truth, predictions)}


In [ ]:
def make_optimizer(model, model_name, stage):
    head_params = list(classifier(model, model_name).parameters())
    head_ids = {id(parameter) for parameter in head_params}
    groups = [{'params': head_params, 'lr': cfg.head_lr if stage == 'head' else cfg.finetune_head_lr}]
    if stage == 'finetune':
        groups.append({'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': cfg.backbone_lr})
    return torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)

def fit_one_stage(model, model_name, stage, epochs, loaders, generator, directory, history,
                  elapsed_total, resume_state=None):
    configure_stage(model, model_name, stage)
    start_epoch, bad_epochs, best_f1 = 0, 0, -1.0
    if resume_state:
        generator.set_state(resume_state['loader_generator'].cpu())
        restore_rng(resume_state['rng'])
        if resume_state['stage_complete']:
            return history, elapsed_total
        start_epoch = resume_state['epoch'] + 1
        bad_epochs, best_f1 = resume_state['bad_epochs'], resume_state['stage_best_f1']
    optimizer = make_optimizer(model, model_name, stage)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    if resume_state:
        optimizer.load_state_dict(resume_state['optimizer'])
        for state in optimizer.state.values():
            for key, value in state.items():
                if torch.is_tensor(value) and key != 'step':
                    state[key] = value.to(device)
        scheduler.load_state_dict(resume_state['scheduler'])
        scaler.load_state_dict(resume_state['scaler'])
    for epoch in range(start_epoch, epochs):
        started = time.perf_counter()
        train_metrics = train_epoch(model, model_name, stage, loaders['train'], optimizer, scaler)
        val_metrics, _, _ = evaluate_neural(model, loaders['val'], device)
        elapsed_total += time.perf_counter() - started
        if val_metrics['macro_f1'] > best_f1:
            best_f1, bad_epochs = val_metrics['macro_f1'], 0
            atomic_torch_save({
                'model': model.state_dict(), 'model_name': model_name, 'stage': stage,
                'epoch': epoch, 'validation_metrics': val_metrics, 'run_signature': RUN_SIGNATURE,
                'class_names': class_names, 'weights': WEIGHTS[model_name].name,
            }, directory / f'best_{stage}.pt')
        else:
            bad_epochs += 1
        history.append({
            'stage': stage, 'epoch': epoch + 1, 'global_epoch': len(history) + 1,
            'head_lr': optimizer.param_groups[0]['lr'],
            **{f'train_{key}': value for key, value in train_metrics.items()},
            **{f'val_{key}': value for key, value in val_metrics.items()},
        })
        scheduler.step()
        stopping = stage == 'finetune' and bad_epochs >= cfg.patience
        stage_complete = stopping or epoch + 1 == epochs
        atomic_torch_save({
            'model': model.state_dict(), 'model_name': model_name, 'run_signature': RUN_SIGNATURE,
            'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
            'stage': stage, 'epoch': epoch, 'bad_epochs': bad_epochs, 'stage_best_f1': best_f1,
            'stage_complete': stage_complete, 'run_complete': stage == 'finetune' and stage_complete,
            'history': history, 'training_seconds': elapsed_total,
            'rng': rng_state(), 'loader_generator': generator.get_state(),
        }, directory / 'last.pt')
        pd.DataFrame(history).to_csv(directory / 'history.csv', index=False)
        print(f"{model_name} | {stage} {epoch + 1}/{epochs} | "
              f"loss train={train_metrics['loss']:.4f} | val macro-F1={val_metrics['macro_f1']:.4f}")
        if stopping:
            print('Early stopping trong fine-tuning: validation macro-F1 không cải thiện.')
            break
    del optimizer, scheduler, scaler
    gc.collect()
    return history, elapsed_total


In [ ]:
def select_stage(head_metrics, finetune_metrics):
    return 'finetune' if finetune_metrics['macro_f1'] > head_metrics['macro_f1'] else 'head'

def train_transfer(model, model_name):
    model.to(device)
    directory = ROOT / model_name
    directory.mkdir(exist_ok=True)
    loaders, generator = make_loaders(model_name)
    resume_state, history, elapsed_total = None, [], 0.0
    last_path = directory / 'last.pt'
    if cfg.resume and last_path.exists():
        resume_state = load_checkpoint(last_path, model_name)
        model.load_state_dict(resume_state['model'])
        history, elapsed_total = resume_state['history'], resume_state['training_seconds']
        required = ['best_head.pt'] + (['best_finetune.pt'] if resume_state['stage'] == 'finetune' else [])
        if not all((directory / filename).exists() for filename in required):
            raise FileNotFoundError('Resume cần các checkpoint best của stage đã chạy cùng với last.pt.')
    head_epochs = 1 if cfg.smoke_test else cfg.head_epochs
    fine_epochs = 1 if cfg.smoke_test else cfg.finetune_epochs
    if resume_state is None or resume_state['stage'] == 'head':
        history, elapsed_total = fit_one_stage(
            model, model_name, 'head', head_epochs, loaders, generator, directory, history,
            elapsed_total, resume_state)
        head_checkpoint = load_checkpoint(directory / 'best_head.pt', model_name)
        model.load_state_dict(head_checkpoint['model'])
        del head_checkpoint
        # Optimizer fine-tuning được tạo mới; generator tiếp tục sau các epoch head.
        resume_state = None
    history, elapsed_total = fit_one_stage(
        model, model_name, 'finetune', fine_epochs, loaders, generator, directory, history,
        elapsed_total, resume_state)
    del resume_state
    stage_rows, metrics = [], {}
    for stage in ('head', 'finetune'):
        saved = load_checkpoint(directory / f'best_{stage}.pt', model_name)
        metrics[stage] = saved['validation_metrics']
        stage_rows.append({'stage': stage, 'best_epoch': saved['epoch'] + 1,
                           'val_loss': metrics[stage]['loss'], 'val_accuracy': metrics[stage]['accuracy'],
                           'val_macro_f1': metrics[stage]['macro_f1']})
        del saved
    selected = select_stage(metrics['head'], metrics['finetune'])
    delta_pp = 100 * (metrics['finetune']['macro_f1'] - metrics['head']['macro_f1'])
    for row in stage_rows:
        row.update(selected=row['stage'] == selected,
                   delta_macro_f1_pp=delta_pp if row['stage'] == 'finetune' else 0.0)
    final_checkpoint = load_checkpoint(directory / f'best_{selected}.pt', model_name)
    model.load_state_dict(final_checkpoint['model'])
    atomic_torch_save(final_checkpoint, directory / 'best.pt')
    del final_checkpoint
    model.eval()
    # Chỉ cần tối đa hai ảnh xác định để kiểm tra reload; không tạo iterator
    # multiprocessing rồi bỏ dở sau batch đầu.
    probe_dataset = SceneDataset(splits['val'], WEIGHTS[model_name].transforms())
    example_x = torch.stack([
        probe_dataset[index][0] for index in range(min(2, len(probe_dataset)))
    ]).to(device)
    del probe_dataset
    with torch.inference_mode():
        expected_logits = model(example_x).cpu()
    with torch.no_grad():
        next(classifier(model, model_name).parameters()).add_(0.5)
    restored = load_checkpoint(directory / 'best.pt', model_name)
    model.load_state_dict(restored['model'])
    del restored
    with torch.inference_mode():
        torch.testing.assert_close(model(example_x).cpu(), expected_logits, atol=1e-6, rtol=1e-5)
    summary = {'model_name': model_name, 'selected_stage': selected, 'stage_rows': stage_rows,
               'delta_macro_f1_pp': delta_pp, 'history': history, 'training_seconds': elapsed_total,
               'checkpoint_reload_check': 'passed', 'run_signature': RUN_SIGNATURE}
    pd.DataFrame(stage_rows).to_csv(directory / 'stage_results.csv', index=False)
    write_json(directory / 'training_summary.json', summary)
    del example_x, loaders
    gc.collect()
    return summary


In [ ]:
MODEL_NAMES = ['mobilenet_v3_small', 'vit_b_16']
if RUN_TRAINING:
    for model_name in MODEL_NAMES:
        seed_everything(cfg.seed)
        resume_model = cfg.resume and (ROOT / model_name / 'last.pt').is_file()
        model = build_model(model_name, pretrained=not resume_model)
        parameter_count = sum(p.numel() for p in model.parameters())
        training = train_transfer(model, model_name)
        training['parameters'] = parameter_count
        write_json(ROOT / model_name / 'training_summary.json', training)
        model_results[model_name] = {'training': training, 'evaluation': None}
        model.to('cpu')
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# (7) Vẽ đường học

Đối chiếu loss/accuracy/macro-F1 train và validation, đánh dấu thời điểm mở backbone. Bảng stage cho biết checkpoint được chọn và mức thay đổi theo điểm phần trăm.


In [ ]:
def delta_description(delta_pp):
    if math.isclose(delta_pp, 0.0, abs_tol=1e-9):
        return 'không đổi (0.00 điểm phần trăm)'
    direction = 'tăng' if delta_pp > 0 else 'giảm'
    return f'{direction} {abs(delta_pp):.2f} điểm phần trăm'

def stage_caption(training):
    delta = training['delta_macro_f1_pp']
    if cfg.smoke_test:
        return (f"SMOKE TEST: Δ validation macro-F1={delta:+.2f} điểm phần trăm; "
                f"checkpoint chọn stage {training['selected_stage']}. "
                'Đã kiểm chứng hai stage, chưa dùng số liệu subset để kết luận chất lượng fine-tuning.')
    return (f"Trong lần chạy seed {cfg.seed}, validation macro-F1 {delta_description(delta)} "
            f"sau fine-tuning. Checkpoint cuối được chọn từ stage {training['selected_stage']} bằng validation.")

def show_stage_results(model_name, training):
    table = pd.DataFrame(training['stage_rows'])
    print(f'\n{model_name}: đối chiếu hai giai đoạn trong cùng mô hình')
    print(table.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
    print('Δ macro-F1 (fine-tuning − head):', f"{training['delta_macro_f1_pp']:+.2f} điểm phần trăm")
    history = pd.DataFrame(training['history'])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric in zip(axes, ['loss', 'accuracy', 'macro_f1']):
        ax.plot(history.global_epoch, history[f'train_{metric}'], label='train có augmentation')
        ax.plot(history.global_epoch, history[f'val_{metric}'], label='validation xác định')
        ax.axvline(int((history.stage == 'head').sum()) + 0.5, linestyle='--', color='gray', label='mở backbone')
        ax.set(xlabel='Epoch', ylabel=metric, title=model_name)
        ax.legend(fontsize=8)
    save_figure(f'{model_name}_learning_curves', stage_caption(training))


In [ ]:
if RUN_TRAINING:
    for model_name, result in model_results.items():
        show_stage_results(model_name, result['training'])


# (8) Đánh giá mô hình

Nạp lại best.pt cho từng mô hình. Train/validation/test đều được đánh giá với preprocessing xác định; lưu xác suất từng lớp và kiểm tra sample_id. Confusion matrix hiển thị số lượng và tỷ lệ theo lớp thật.


In [ ]:
def error_summary(matrix):
    off_diagonal = matrix.copy()
    np.fill_diagonal(off_diagonal, 0)
    if int(off_diagonal.max()) == 0:
        return None
    true_label, predicted_label = np.unravel_index(off_diagonal.argmax(), off_diagonal.shape)
    count = int(off_diagonal[true_label, predicted_label])
    support = int(matrix[true_label].sum())
    return {'true_class': class_names[true_label], 'predicted_class': class_names[predicted_label],
            'count': count, 'true_class_support': support, 'percent_of_true_class': 100 * count / support}

def error_description(error, n_test):
    if error is None:
        return f'Không thấy dự đoán sai trong {n_test} ảnh test của lần chạy này.'
    return (f"Nhầm nhiều nhất: {error['true_class']} → {error['predicted_class']}, "
            f"{error['count']}/{error['true_class_support']} ảnh lớp thật "
            f"({error['percent_of_true_class']:.2f}%).")

def evaluation_caption(evaluation):
    values = evaluation['metrics']
    gap = 100 * (values['train']['macro_f1'] - values['val']['macro_f1'])
    prefix = 'SMOKE TEST, chỉ kiểm chứng kỹ thuật. ' if cfg.smoke_test else f'Lần chạy seed {cfg.seed}. '
    return (prefix + f"Test accuracy={100 * values['test']['accuracy']:.2f}%, "
            f"macro-F1={values['test']['macro_f1']:.4f}; "
            f'Δ macro-F1 train − validation={gap:+.2f} điểm phần trăm. ' +
            error_description(evaluation['error'], len(splits['test'])))


In [ ]:
def image_probability_metrics(labels, probabilities):
    return {
        'top2_accuracy': float(top_k_accuracy_score(labels, probabilities, k=2, labels=LABELS)),
        'top3_accuracy': float(top_k_accuracy_score(labels, probabilities, k=3, labels=LABELS)),
        'macro_roc_auc_ovr': float(roc_auc_score(
            labels, probabilities, labels=LABELS, multi_class='ovr', average='macro'
        )),
    }


def evaluate_final_model(model, model_name):
    model.to(device)
    model.eval()
    directory = ROOT / model_name
    metrics, per_class_tables = {}, []
    test_predictions, test_matrix = None, None
    for split in ('train', 'val', 'test'):
        frame = splits[split]
        loader = DataLoader(SceneDataset(frame, WEIGHTS[model_name].transforms()),
                            batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers,
                            persistent_workers=False,
                            pin_memory=device.type == 'cuda', worker_init_fn=seed_worker,
                            generator=torch.Generator().manual_seed(cfg.seed))
        values, predictions, split_probabilities = evaluate_neural(model, loader, device)
        assert predictions.sample_id.tolist() == frame.sample_id.tolist()
        np.testing.assert_array_equal(predictions.label.to_numpy(), frame.label.to_numpy())
        report = classification_report(predictions.label, predictions.prediction, labels=LABELS,
                                       target_names=class_names, output_dict=True, zero_division=0)
        values.update(
            weighted_precision=float(report['weighted avg']['precision']),
            weighted_recall=float(report['weighted avg']['recall']),
            support=len(predictions),
            **image_probability_metrics(predictions.label, split_probabilities),
        )
        per_class = (pd.DataFrame(report).T.loc[class_names]
                     .rename_axis('class_name').reset_index()
                     .rename(columns={'f1-score': 'f1'}))
        per_class.insert(0, 'split', split)
        per_class['support'] = per_class.support.astype(int)
        per_class_tables.append(per_class)
        write_json(directory / f'{split}_metrics.json', values)
        write_json(directory / f'{split}_classification_report.json', report)
        pd.DataFrame(report).T.to_csv(directory / f'{split}_classification_report.csv')
        predictions['true_class'] = predictions.label.map(dict(enumerate(class_names)))
        predictions['predicted_class'] = predictions.prediction.map(dict(enumerate(class_names)))
        if split == 'test':
            for label, name in enumerate(class_names):
                predictions[f'prob_{name}'] = split_probabilities[:, label]
            predictions['confidence'] = split_probabilities.max(axis=1)
            test_predictions = predictions
            test_matrix = confusion_matrix(predictions.label, predictions.prediction, labels=LABELS)
            assert test_matrix.shape == (6, 6) and int(test_matrix.sum()) == len(frame)
            assert math.isclose(float(np.trace(test_matrix) / test_matrix.sum()), values['accuracy'])
            pd.DataFrame(test_matrix, index=class_names, columns=class_names).to_csv(directory / 'confusion_counts.csv')
            normalized = test_matrix / test_matrix.sum(axis=1, keepdims=True)
            pd.DataFrame(normalized, index=class_names, columns=class_names).to_csv(directory / 'confusion_normalized.csv')
        predictions.to_csv(directory / f'{split}_predictions.csv', index=False)
        metrics[split] = values
        del loader
    print(f'\n{model_name}: checkpoint cuối, preprocessing xác định cho mọi split')
    summary = pd.DataFrame.from_dict(metrics, orient='index').rename_axis('split')
    per_class = pd.concat(per_class_tables, ignore_index=True)
    summary.to_csv(directory / 'overall_metrics.csv')
    per_class.to_csv(directory / 'per_class_metrics.csv', index=False)
    display(summary.round(4))
    display(per_class.round(4))
    evaluation = {'metrics': metrics, 'per_class': per_class, 'test_predictions': test_predictions,
                  'test_matrix': test_matrix,
                  'error': error_summary(test_matrix)}
    write_json(directory / 'evaluation_summary.json', {'metrics': metrics, 'error': evaluation['error']})
    gc.collect()
    return evaluation


In [ ]:
def plot_final_results(model_name, evaluation):
    caption = evaluation_caption(evaluation)
    table = pd.DataFrame(evaluation['metrics']).T
    table[['accuracy', 'macro_f1', 'balanced_accuracy']].plot.bar(figsize=(9, 4), rot=0, ylim=(0, 1))
    plt.title(f'{model_name}: checkpoint cuối trên ba split')
    plt.ylabel('Giá trị metric')
    save_figure(f'{model_name}_split_metrics', caption)
    matrix = evaluation['test_matrix']
    normalized = matrix / matrix.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, values, fmt, title in zip(axes, [matrix, normalized], ['d', '.2f'], ['Số lượng', 'Tỷ lệ theo lớp thật']):
        sns.heatmap(values, annot=True, fmt=fmt, xticklabels=class_names, yticklabels=class_names,
                    cmap='Blues', ax=ax, vmin=0, vmax=1 if fmt == '.2f' else None)
        ax.set(title=f'{model_name}: {title}', xlabel='Dự đoán', ylabel='Nhãn thật')
    save_figure(f'{model_name}_confusion', caption)


In [ ]:
if RUN_TRAINING:
    for model_name, result in model_results.items():
        seed_everything(cfg.seed)
        model = build_model(model_name, pretrained=False)
        checkpoint = load_checkpoint(ROOT / model_name / 'best.pt', model_name)
        model.load_state_dict(checkpoint['model'])
        del checkpoint
        evaluation = evaluate_final_model(model, model_name)
        result['evaluation'] = evaluation
        plot_final_results(model_name, evaluation)
        model.to('cpu')
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# (9) So sánh mô hình

Hai mô hình dùng cùng test và cùng tiêu chí checkpoint. Đọc macro-F1 cùng recall từng lớp, số tham số và thời gian huấn luyện; một seed chưa đủ để kết luận hơn kém trong mọi lần chạy.


In [ ]:
if RUN_TRAINING:
    comparison = pd.DataFrame([
        {'model': name, **result['evaluation']['metrics']['test'],
         'parameters': result['training']['parameters'],
         'training_seconds': result['training']['training_seconds'],
         'selected_stage': result['training']['selected_stage']}
        for name, result in model_results.items()])
    comparison.to_csv(ROOT / 'model_comparison.csv', index=False)
    write_json(ROOT / 'model_comparison.json', comparison.to_dict('records'))
    display(comparison.round(4))
    ax = comparison.set_index('model')[['accuracy', 'macro_precision', 'macro_recall', 'macro_f1']].plot.bar(
        figsize=(10, 4), rot=0, ylim=(0, 1))
    ax.set(ylabel='Test metric', xlabel='')
    best = comparison.sort_values('macro_f1', ascending=False).iloc[0]
    save_figure('model_comparison', f"Seed {cfg.seed}: {best.model} có test macro-F1 cao nhất {100 * best.macro_f1:.2f}%.")


# (10) Phân tích mẫu sai

Chọn tối đa 12 ảnh sai tự tin nhất, có thứ tự xác định; hiển thị đúng ảnh sau preprocessing và nhãn thật/dự đoán. Đọc các cặp nhãn nhầm từ ma trận, không tự sửa nhãn theo confidence.


In [ ]:
def display_rgb(input_tensor):
    mean = torch.tensor([.485, .456, .406])[:, None, None]
    std = torch.tensor([.229, .224, .225])[:, None, None]
    return (input_tensor.detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


if RUN_TRAINING:
    lookup = splits['test'].set_index('sample_id')
    for name, result in model_results.items():
        predictions = result['evaluation']['test_predictions']
        wrong = predictions.loc[predictions.label != predictions.prediction].sort_values(
            ['confidence', 'sample_id'], ascending=[False, True])
        wrong.to_csv(ROOT / name / 'errors.csv', index=False)
        print(name, ':', len(wrong), '/', len(predictions), 'ảnh sai')
        if wrong.empty:
            print('Không có ảnh sai để hiển thị.')
            continue
        selected = wrong.head(12)
        fig, axes = plt.subplots(math.ceil(len(selected) / 4), 4,
                                 figsize=(14, 3.5 * math.ceil(len(selected) / 4)), squeeze=False)
        for ax in axes.flat:
            ax.axis('off')
        for ax, row in zip(axes.flat, selected.itertuples()):
            tensor = WEIGHTS[name].transforms()(image_from_row(lookup.loc[row.sample_id]))
            ax.imshow(display_rgb(tensor))
            ax.set_title(f'{row.sample_id}\nThật {row.true_class} → {row.predicted_class}\np={row.confidence:.3f}', fontsize=9)
        save_figure(f'{name}_errors', error_description(result['evaluation']['error'], len(predictions)))


# (11) Giải thích bằng Grad-CAM và kết luận

Dùng MobileNet với lớp features[-1]. Chọn tối đa hai ảnh đúng và hai ảnh sai từ dự đoán thật. Heatmap chồng lên đúng ảnh crop mà mô hình nhận; màu nóng biểu diễn vùng đóng góp vào lớp dự đoán, không phải mask phân đoạn. Nguồn: [pytorch-grad-cam](https://github.com/jacobgil/pytorch-grad-cam).


In [ ]:
if RUN_TRAINING:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image
    name = 'mobilenet_v3_small'
    model = build_model(name, pretrained=False).to(device).eval()
    model.load_state_dict(load_checkpoint(ROOT / name / 'best.pt', name)['model'])
    def weights_digest(model):
        digest = hashlib.sha256()
        for tensor in model.state_dict().values():
            digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
        return digest.hexdigest()
    digest_before = weights_digest(model)
    rows = model_results[name]['evaluation']['test_predictions']
    selected = pd.concat([rows.loc[(rows.label == rows.prediction) == correct]
                          .sort_values('sample_id').head(2) for correct in [True, False]])
    selected.to_csv(ROOT / name / 'gradcam_samples.csv', index=False)
    lookup = splits['test'].set_index('sample_id')
    if selected.empty:
        print('Không có mẫu phù hợp để vẽ Grad-CAM.')
    else:
        fig, axes = plt.subplots(len(selected), 2, figsize=(8, 4 * len(selected)), squeeze=False)
        with GradCAM(model=model, target_layers=[model.features[-1]]) as cam:
            for axes_row, row in zip(axes, selected.itertuples()):
                tensor = WEIGHTS[name].transforms()(image_from_row(lookup.loc[row.sample_id]))
                # Input có gradient kể cả nếu mọi trọng số backbone đang frozen.
                input_tensor = tensor.unsqueeze(0).to(device).requires_grad_(True)
                with torch.enable_grad():
                    heatmap = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(int(row.prediction))])[0]
                assert heatmap.shape == (224, 224) and np.isfinite(heatmap).all()
                rgb = display_rgb(tensor).astype(np.float32)
                axes_row[0].imshow(rgb)
                axes_row[1].imshow(show_cam_on_image(rgb, heatmap, use_rgb=True))
                axes_row[0].set_title(f'{row.sample_id}\nThật: {row.true_class}')
                axes_row[1].set_title(f'Grad-CAM: {row.predicted_class}\np={row.confidence:.3f}')
                for ax in axes_row:
                    ax.axis('off')
        save_figure('mobilenet_v3_small_gradcam', 'Ảnh gốc trong hình là đúng crop 224×224 mô hình nhận; heatmap nhắm lớp dự đoán.')
    assert weights_digest(model) == digest_before
    model.zero_grad(set_to_none=True)
    model.to('cpu')
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
def print_model_conclusion(model_name, result):
    training = result['training']
    by_stage = {row['stage']: row for row in training['stage_rows']}
    head, fine = by_stage['head'], by_stage['finetune']
    mode = 'SMOKE TEST — chỉ kiểm chứng kỹ thuật' if cfg.smoke_test else f'FULL — seed {cfg.seed}'
    print(f'\n=== {model_name} | {mode} ===')
    print(f"Số ảnh: train={len(splits['train'])}, validation={len(splits['val'])}, test={len(splits['test'])}.")
    print('Câu hỏi 1: Fine-tuning backbone có cải thiện validation so với chỉ train classifier không?')
    print(f"  Head: epoch {head['best_epoch']}, loss={head['val_loss']:.4f}, "
          f"accuracy={100 * head['val_accuracy']:.2f}%, macro-F1={head['val_macro_f1']:.4f}.")
    print(f"  Fine-tuning: epoch {fine['best_epoch']}, loss={fine['val_loss']:.4f}, "
          f"accuracy={100 * fine['val_accuracy']:.2f}%, macro-F1={fine['val_macro_f1']:.4f}.")
    print('  Kết luận:', stage_caption(training))
    print('Câu hỏi 2: Checkpoint cuối phân loại test tốt đến đâu, và còn nhầm ở những lớp nào?')
    evaluation = result.get('evaluation')
    if evaluation is None:
        print('  Chưa chạy cell đánh giá cuối của phần này; chưa có kết luận trên test.')
        return
    metrics = evaluation['metrics']
    for split in ('train', 'val', 'test'):
        values = metrics[split]
        print(f"  {split}: loss={values['loss']:.4f}, accuracy={100 * values['accuracy']:.2f}%, "
              f"macro-F1={values['macro_f1']:.4f}, balanced accuracy={100 * values['balanced_accuracy']:.2f}%.")
    print('  Kết luận:', evaluation_caption(evaluation))
    if not cfg.smoke_test:
        print('  Khoảng cách train–validation cần đọc cùng đường học và lỗi từng lớp; một lần chạy chưa đánh giá độ ổn định.')
    print(f'  Test giữ {test_duplicates} bản sao chính xác ngoài ảnh đầu tiên mỗi nhóm; hash chưa bao phủ near-duplicate.')


In [ ]:
def verify_model_result(model_name, result):
    training = result['training']
    stage_metrics = {}
    for row in training['stage_rows']:
        saved = load_checkpoint(ROOT / model_name / f"best_{row['stage']}.pt", model_name)
        values = saved['validation_metrics']
        stage_metrics[row['stage']] = values
        assert row['best_epoch'] == saved['epoch'] + 1
        for metric in ('loss', 'accuracy', 'macro_f1'):
            assert math.isclose(row[f'val_{metric}'], values[metric], abs_tol=1e-12)
        history_row = next(h for h in training['history'] if h['stage'] == row['stage'] and h['epoch'] == row['best_epoch'])
        assert math.isclose(history_row['val_macro_f1'], row['val_macro_f1'], abs_tol=1e-12)
        del saved
    assert training['selected_stage'] == select_stage(stage_metrics['head'], stage_metrics['finetune'])
    expected_delta = 100 * (stage_metrics['finetune']['macro_f1'] - stage_metrics['head']['macro_f1'])
    assert math.isclose(expected_delta, training['delta_macro_f1_pp'], abs_tol=1e-12)
    if result.get('evaluation') is not None:
        evaluation = result['evaluation']
        predictions = pd.read_csv(ROOT / model_name / 'test_predictions.csv', float_precision='round_trip')
        assert predictions.sample_id.tolist() == splits['test'].sample_id.tolist()
        np.testing.assert_array_equal(predictions.label.to_numpy(), splits['test'].label.to_numpy())
        values = metric_values(predictions.label, predictions.prediction)
        values.update(image_probability_metrics(
            predictions.label, predictions[[f'prob_{name}' for name in class_names]].to_numpy()
        ))
        for key, value in values.items():
            assert math.isclose(value, evaluation['metrics']['test'][key], abs_tol=1e-12)
        matrix = confusion_matrix(predictions.label, predictions.prediction, labels=LABELS)
        np.testing.assert_array_equal(matrix, evaluation['test_matrix'])
        assert int(matrix.sum()) == len(splits['test'])
    print(f'{model_name}: checkpoint, stage metrics và kết quả hiện có nhất quán.')


if RUN_TRAINING:
    assert set(model_results) == set(MODEL_NAMES)
    for model_name, result in model_results.items():
        verify_model_result(model_name, result)
        print_model_conclusion(model_name, result)
    print('Kết luận so sánh:')
    print(comparison.sort_values('macro_f1', ascending=False).to_string(index=False))
    write_json(ROOT / 'verification.json', {'models': MODEL_NAMES, 'status': 'passed', 'smoke_test': cfg.smoke_test})
else:
    print('Đã xuất EDA; không huấn luyện hay đánh giá mô hình.')
